# Experiment 10b - Expanded temporal dynamics (block C) + leaderboard submission

10a found block C (temporal dynamics) was the only feature direction that helped, and
it helped on **all three folds** - but at +0.038 CV-WAPE it sat just under the 0.05
bar. This notebook (1) gives C a **fair, expanded** chance (more velocities,
acceleration, multi-span EWMA, regime + spike flags), re-checks it on CV, then (2)
builds a **leaderboard submission** with it.

**Submission design (honest):** C only moves the LGBM arm, so we don't submit LGBM
alone (14.78 - worse than your 14.68). Instead we put C on the LGBM arm and **blend
with your already-saved gated-MLP validation predictions** (`val_components_7c.csv`)
- the same 14.68 ensemble, now with a C-augmented LGBM arm. No MLP retrain, CPU-only.
Blend weight is re-optimised on OOF. Expectation: C's +0.038 LGBM gain is diluted by
the LGBM arm's ~0.285 weight, so the ensemble likely lands ~14.66-14.68; the
leaderboard is the real test of whether C's CV signal is genuine.


---
## §0. Configuration

In [ ]:
from pathlib import Path
DATA_DIR   = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR = Path("/kaggle/working")
# Local:
# DATA_DIR   = Path("data"); OUTPUT_DIR = Path("experiment-10")

# saved gated-MLP artifacts (upload on Kaggle). Local paths in comments.
MLP_VAL_PATH = Path("/kaggle/input/exp7-oof/val_components_7c.csv")          # has series_id,timestamp,pred_mlp
MLP_OOF_PATH = Path("/kaggle/input/exp7-oof/oof_predictions_7e_mlp_gated.csv")# series_id,timestamp,target,pred_mlp_gated,fold
# Local: experiment-7/val_components_7c.csv  /  experiment-7/oof_predictions_7e_mlp_gated.csv

TRAIN_PATH = DATA_DIR/"train.csv"; VAL_INPUT_PATH = DATA_DIR/"validation_input.csv"
SEED, N_FOLDS, EVAL_HORIZON = 42, 3, 336
SEEDS = [42, 101, 202, 303, 404]   # 5-seed production convention

DOMAIN_FEAT_COLS = ["workload_intensity","demand_forecast","upstream_quality_forecast",
    "promotion_intensity","shock_risk","maintenance_known","unit_reliability_forecast",
    "queue_pressure_forecast","network_pressure_forecast","event_load_forecast",
    "service_irregularity_risk_forecast"]
TIME_COLS=["hour_sin","hour_cos","dow_sin","dow_cos","doy_sin","doy_cos","is_weekend"]
TOP_DOM=["queue_pressure_forecast","network_pressure_forecast","workload_intensity","shock_risk","event_load_forecast"]
TOP3=TOP_DOM[:3]; LAGS,LEADS,WINS=[1,24],[1,24],[6,24]
DYN_COVS=TOP_DOM                       # expanded: dynamics for all 5 stress covariates
BASELINE_LGBM=dict(n_estimators=600,learning_rate=0.03,num_leaves=63,subsample=0.8,
    colsample_bytree=0.8,min_child_samples=50,reg_lambda=1.0,verbose=-1)
ACCEPT_DELTA=0.05; W_LGBM_7C=0.285     # the shipped ensemble weight on the LGBM arm
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print("expanded C dynamics on", len(DYN_COVS), "covariates; 5-seed production; submission via saved MLP blend")


---
## §1. Setup + load train & validation_input

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, lightgbm as lgb
np.random.seed(SEED)
def wape(y,p,eps=1e-8):
    y,p=np.asarray(y,float),np.asarray(p,float); return np.sum(np.abs(y-p))/(np.sum(np.abs(y))+eps)*100
train_df=pd.read_csv(TRAIN_PATH,parse_dates=["timestamp"])
val_input=pd.read_csv(VAL_INPUT_PATH,parse_dates=["timestamp"])
sid_to_code={s:i for i,s in enumerate(sorted(train_df.series_id.unique()))}
print("train",train_df.shape,"| val_input",val_input.shape)


---
## §2. Feature builders - baseline pipeline + EXPANDED block C (causal)

In [ ]:
def impute(df,cols):
    df=df.sort_values(["series_id","timestamp"]).copy()
    df[cols]=df.groupby("series_id",group_keys=False)[cols].transform(lambda x:x.ffill().bfill()); return df
def add_time(df):
    df=df.copy(); t=pd.to_datetime(df.timestamp)
    df["hour_sin"]=np.sin(2*np.pi*t.dt.hour/24); df["hour_cos"]=np.cos(2*np.pi*t.dt.hour/24)
    df["dow_sin"]=np.sin(2*np.pi*t.dt.dayofweek/7); df["dow_cos"]=np.cos(2*np.pi*t.dt.dayofweek/7)
    df["doy_sin"]=np.sin(2*np.pi*t.dt.dayofyear/365); df["doy_cos"]=np.cos(2*np.pi*t.dt.dayofyear/365)
    df["is_weekend"]=(t.dt.dayofweek>=5).astype(float); return df
def add_temporal(df):
    df=df.copy().sort_values(["series_id","timestamp"]); new=[]
    for c in TOP_DOM:
        g=df.groupby("series_id",sort=False)[c]
        for L in LAGS: k=f"{c}_lag{L}"; df[k]=g.shift(L); new.append(k)
        for L in LEADS: k=f"{c}_lead{L}"; df[k]=g.shift(-L); new.append(k)
    for c in TOP3:
        g=df.groupby("series_id",sort=False)[c]
        for W in WINS:
            km=f"{c}_roll{W}m"; df[km]=g.shift(1).transform(lambda x:x.rolling(W,min_periods=1).mean()); new.append(km)
            ks=f"{c}_roll{W}s"; df[ks]=g.shift(1).transform(lambda x:x.rolling(W,min_periods=1).std().fillna(0)); new.append(ks)
    return df,new

def add_dynamics_expanded(df):
    "EXPANDED block C - all causal (shift/diff/expanding). Safe to compute across train|val boundary."
    df=df.copy().sort_values(["series_id","timestamp"]); new=[]
    for c in DYN_COVS:                         # velocities for all 5 stress covariates
        g=df.groupby("series_id",sort=False)[c]
        df[f"{c}_vel1"]=g.diff(1);  new.append(f"{c}_vel1")
        df[f"{c}_vel24"]=g.diff(24); new.append(f"{c}_vel24")
    for c in TOP3:                             # acceleration + multi-span EWMA for top 3
        g=df.groupby("series_id",sort=False)[c]
        df[f"{c}_acc1"]=g.diff(1).groupby(df.series_id,sort=False).diff(1); new.append(f"{c}_acc1")
        for span in (6,12,24):
            df[f"{c}_ewm{span}"]=g.transform(lambda x:x.shift(1).ewm(span=span,min_periods=1).mean()); new.append(f"{c}_ewm{span}")
    for c in ["queue_pressure_forecast","network_pressure_forecast"]:   # regime + spike flags
        g=df.groupby("series_id",sort=False)[c]
        roll24=g.transform(lambda x:x.shift(1).rolling(24,min_periods=1).mean())
        expmed=g.transform(lambda x:x.shift(1).expanding(min_periods=1).median())
        df[f"{c}_regime"]=(roll24>expmed).astype(float); new.append(f"{c}_regime")
        p90=g.transform(lambda x:x.shift(1).expanding(min_periods=24).quantile(0.9))
        df[f"{c}_spike"]=(df[c]>p90).astype(float); new.append(f"{c}_spike")
    return df,new

# build the CV frame (train only)
base=add_time(impute(train_df,DOMAIN_FEAT_COLS)); base["sid_code"]=base.series_id.map(sid_to_code).astype("int32")
base,TEMP=add_temporal(base); base,DYN=add_dynamics_expanded(base)
FEATS_BASE=DOMAIN_FEAT_COLS+TIME_COLS+["series_mean"]+TEMP+["sid_code"]
FEATS_C=FEATS_BASE+DYN
print(f"baseline {len(FEATS_BASE)} feats | expanded C adds {len(DYN)}: {DYN}")


---
## §3. CV: baseline vs +expanded-C (give C its fair chance)

In [ ]:
def fold_off(k): return (N_FOLDS-k)*EVAL_HORIZON
def fold_split(end_off,length=EVAL_HORIZON):
    tr_l,te_l=[],[]
    for _,g in base.groupby("series_id",sort=False):
        g=g.sort_values("timestamp"); cut=len(g)-end_off; tr_l.append(g.iloc[:cut]); te_l.append(g.iloc[cut:cut+length])
    tr=pd.concat(tr_l,ignore_index=True); te=pd.concat(te_l,ignore_index=True)
    sm=tr.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
    return tr.merge(sm,on="series_id"),te.merge(sm,on="series_id")
def fit_pred(tr,te,feats,seeds=[42,101,202]):
    ps=[]
    for s in seeds:
        m=lgb.LGBMRegressor(**BASELINE_LGBM,objective="regression_l1",random_state=s)
        m.fit(tr[feats],tr.target.to_numpy(float),categorical_feature=["sid_code"]); ps.append(m.predict(te[feats]))
    return np.clip(np.mean(ps,axis=0),0,None)

oof_c_rows=[]; pf_base=[]; pf_c=[]
for k in range(N_FOLDS):
    tr,te=fold_split(fold_off(k))
    pb=fit_pred(tr,te,FEATS_BASE); pc=fit_pred(tr,te,FEATS_C)
    pf_base.append(wape(te.target,pb)); pf_c.append(wape(te.target,pc))
    oof_c_rows.append(te[["series_id","timestamp","target"]].assign(pred_lgbm_c=pc,fold=k))
oof_c=pd.concat(oof_c_rows,ignore_index=True)
b,c=np.mean(pf_base),np.mean(pf_c)
print(f"baseline      : folds {[round(x,3) for x in pf_base]}  mean {b:.3f}")
print(f"+expanded C   : folds {[round(x,3) for x in pf_c]}  mean {c:.3f}")
print(f"Δ vs baseline : {b-c:+.3f}  ({'ACCEPT (>'+str(ACCEPT_DELTA)+')' if b-c>ACCEPT_DELTA else '~noise'})")


---
## §4. Re-optimise the ensemble blend on OOF (LGBM-C arm + saved MLP OOF)

In [ ]:
mlp_oof=pd.read_csv(MLP_OOF_PATH,parse_dates=["timestamp"])[["series_id","timestamp","fold","pred_mlp_gated"]]
J=oof_c.merge(mlp_oof,on=["series_id","timestamp","fold"],how="inner")
assert len(J)==len(oof_c), "OOF merge dropped rows - check MLP OOF alignment"
pa=J.pred_mlp_gated.to_numpy(); pb=J.pred_lgbm_c.to_numpy()
def cvw(w):
    J["_b"]=w*pa+(1-w)*pb; v=np.mean([wape(g.target,g["_b"]) for _,g in J.groupby("fold")]); J.drop(columns="_b",inplace=True); return float(v)
grid=np.round(np.arange(0,1.0001,0.005),3)
w_star,best=min(((w,cvw(w)) for w in grid),key=lambda t:t[1])
print(f"MLP solo {cvw(1.0):.3f} | LGBM-C solo {cvw(0.0):.3f}")
print(f"shipped weight (w_mlp={1-W_LGBM_7C:.3f}) -> {cvw(1-W_LGBM_7C):.3f}")
print(f"re-optimised   (w_mlp={w_star:.3f})        -> {best:.3f}   <- use this for the submission")


---
## §5. Production: combined-frame FE, full-train LGBM-C, predict validation, blend

In [ ]:
val_stub=val_input.copy(); val_stub["target"]=np.nan
comb=pd.concat([train_df,val_stub],ignore_index=True)
comb["is_val"]=np.r_[np.zeros(len(train_df),bool),np.ones(len(val_stub),bool)]
comb=add_time(impute(comb,DOMAIN_FEAT_COLS)); comb["sid_code"]=comb.series_id.map(sid_to_code).astype("int32")
comb,_=add_temporal(comb); comb,_=add_dynamics_expanded(comb)
sm=comb[~comb.is_val].groupby("series_id")["target"].mean().rename("series_mean").reset_index()
comb=comb.merge(sm,on="series_id")
tr_full=comb[~comb.is_val]; va=comb[comb.is_val].sort_values(["series_id","timestamp"])

# 5-seed LGBM-C on full train
preds=[]
for s in SEEDS:
    m=lgb.LGBMRegressor(**BASELINE_LGBM,objective="regression_l1",random_state=s)
    m.fit(tr_full[FEATS_C],tr_full.target.to_numpy(float),categorical_feature=["sid_code"]); preds.append(m.predict(va[FEATS_C]))
pred_lgbm_c=np.clip(np.mean(preds,axis=0),0,None)
sub=va[["series_id","timestamp"]].copy(); sub["pred_lgbm_c"]=pred_lgbm_c

# blend with saved gated-MLP validation predictions
mlp_val=pd.read_csv(MLP_VAL_PATH,parse_dates=["timestamp"])
mlp_col="pred_mlp" if "pred_mlp" in mlp_val.columns else "prediction"
sub=sub.merge(mlp_val[["series_id","timestamp",mlp_col]].rename(columns={mlp_col:"pred_mlp"}),on=["series_id","timestamp"],how="left")
assert sub.pred_mlp.notna().all(), "MLP val preds did not align to all rows"
sub["prediction"]=w_star*sub.pred_mlp+(1-w_star)*sub.pred_lgbm_c
print("rows",len(sub),"| any NaN:",sub.prediction.isna().any())
print(sub[["series_id","timestamp","pred_lgbm_c","pred_mlp","prediction"]].head())


---
## §6. Write submission + summary

In [ ]:
sub_out=sub[["series_id","timestamp","prediction"]].sort_values(["series_id","timestamp"])
path=OUTPUT_DIR/"submission_10b_ensemble_C.csv"; sub_out.to_csv(path,index=False)
oof_c.to_csv(OUTPUT_DIR/"oof_predictions_10b_lgbm_c.csv",index=False)
print(f"submission -> {path}  ({len(sub_out)} rows)")
print("\n=== Summary ===")
print(f"  expanded-C on LGBM : {c:.3f}  (baseline {b:.3f}, Δ {b-c:+.3f})")
print(f"  ensemble (re-opt)  : OOF {best:.3f} at w_mlp={w_star:.3f}")
print(f"  shipped 7c ensemble: LB 14.68 (CV 14.47) - this submission's CV is {best:.3f}")
print("  -> submit submission_10b_ensemble_C.csv; the LB delta vs 14.68 is the real test of block C.")
